In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Paper Results V1 — fixed eight-source confirmation Run-all

Run this notebook in a Colab GPU runtime. It attempts only `confirm_01` through `confirm_08`; the two development pilots remain outside confirmation. `CENTERED_U` is the frozen primary receiver, while `RAW_U` and `CENTERED_D4` are controls. Every failed, missing, unresolved, or unsupported row remains in the fixed denominator.

`SOURCE_REF` in the next cell is an ordinary editable GitHub locator. The notebook downloads the public companion ZIP when no usable source directory exists and never treats a source digest, manifest, clean checkout, exact package version, or GPU model name as an admission gate.


In [ ]:
# Fixed scope and ordinary companion source. This cell imports only the standard library.
from pathlib import Path
import datetime, importlib, json, os, shutil, signal, subprocess, sys, time, traceback, urllib.request, uuid, zipfile

PILOT_CASES = ()
CONFIRMATION_CASES = ("confirm_01", "confirm_02", "confirm_03", "confirm_04", "confirm_05", "confirm_06", "confirm_07", "confirm_08")
PHASE_ORDER = (
    "generate", "decode", "framewise", "baseline-embed-videoseal",
    "baseline-embed-rivagan", "codec", "quality", "baseline-extract-videoseal",
    "baseline-extract-rivagan", "receiver-sync", "receiver-read",
)
SOURCE_REPOSITORY = "https://github.com/RICHAAARC/SC-SSTW"
SOURCE_REF = "dev/paper-results-v1"
COMPANION_ZIP_URL = SOURCE_REPOSITORY + "/raw/refs/heads/" + SOURCE_REF + "/notebooks/paper_results_v1_companion.zip"
FIXED_FULL_DENOMINATOR = {
    "cases": 10, "artifacts": 690, "receiver_slots": 1600,
    "baseline_slots": 180, "comparison_slots": 180,
    "quality_rows": 70, "cost_rows": 110,
}
FIXED_ATTEMPT_SCOPE = {
    "cases": 8, "case_ids": list(CONFIRMATION_CASES),
    "main_receiver_rows": 2880, "main_receiver_bits": 92160,
    "baseline_rows": 240, "quality_rows": 56,
    "wan_physical_encode_unconditional_range": [0, 2880],
    "preferred_m05_k0_centered_u_rows": 120,
}
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ID = STAMP + "-" + uuid.uuid4().hex[:8]
OUTPUT_PARENT = Path("/content/drive/MyDrive/Video-WM/Paper-Results-V1-Confirmation-V1")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
ACTIVE_RUN_POINTER = OUTPUT_PARENT / "active_confirmation_run.json"
RESUMED_EXISTING_RUN = ACTIVE_RUN_POINTER.is_file()
if RESUMED_EXISTING_RUN:
    RUN_ID = json.loads(ACTIVE_RUN_POINTER.read_text())["run_id"]
OUTPUT_ROOT = OUTPUT_PARENT / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=RESUMED_EXISTING_RUN)
RUN_OUTPUT = OUTPUT_ROOT / "run_state"
PORTABLE_ROOT = Path("/content/paper-results-v1-source")
CACHE_ROOT = Path("/content/drive/MyDrive/Video-WM/Paper-Results-V1-Cache")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_ROOT / "execution.log"
STAGE_RECEIPTS = OUTPUT_ROOT / "stage_receipts.json"
PYTHON = sys.executable

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)

stage_receipts = json.loads(STAGE_RECEIPTS.read_text()) if STAGE_RECEIPTS.is_file() else []
atomic_json(ACTIVE_RUN_POINTER, {"run_id": RUN_ID, "output_root": str(OUTPUT_ROOT)})
def record_stage(stage, status, **fields):
    row = {"stage": stage, "status": status, "recorded_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), **fields}
    stage_receipts.append(row); atomic_json(STAGE_RECEIPTS, stage_receipts); return row

def logged(command, stage, *, cwd=None, env=None, check=True):
    started = time.monotonic(); child = None; returncode = None; primary = None; primary_tb = None; cleanup_errors = []
    print(f"[{stage}] start; full output: {LOG_PATH}", flush=True)
    try:
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("COMMAND " + repr(command) + "\n"); log.flush()
            child = subprocess.Popen(
                command, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT,
                text=True, start_new_session=(os.name == "posix"),
            )
            try:
                returncode = child.wait()
            except BaseException as exc:
                primary = exc; primary_tb = exc.__traceback__
            finally:
                if child is not None and os.name == "posix":
                    try: os.killpg(child.pid, signal.SIGTERM)
                    except ProcessLookupError: pass
                    except BaseException as exc: cleanup_errors.append("group SIGTERM: " + repr(exc))
                    deadline = time.monotonic() + 5.0
                    while time.monotonic() < deadline:
                        try: os.killpg(child.pid, 0)
                        except ProcessLookupError: break
                        except BaseException as exc:
                            cleanup_errors.append("group probe: " + repr(exc)); break
                        time.sleep(0.05)
                    try: os.killpg(child.pid, signal.SIGKILL)
                    except ProcessLookupError: pass
                    except BaseException as exc: cleanup_errors.append("group SIGKILL: " + repr(exc))
                elif child is not None and child.poll() is None:
                    try: child.terminate()
                    except ProcessLookupError: pass
                    except BaseException as exc: cleanup_errors.append("terminate: " + repr(exc))
                if child is not None:
                    try: returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try: child.kill()
                        except ProcessLookupError: pass
                        except BaseException as exc: cleanup_errors.append("kill: " + repr(exc))
                        try: returncode = child.wait(timeout=10)
                        except BaseException as exc: cleanup_errors.append("wait after kill: " + repr(exc))
                    except BaseException as exc: cleanup_errors.append("wait: " + repr(exc))
    except BaseException as exc:
        if primary is None:
            primary = exc; primary_tb = exc.__traceback__
        else:
            cleanup_errors.append("log wrapper: " + repr(exc))
    if primary is None and cleanup_errors:
        primary = RuntimeError("process cleanup failed: " + "; ".join(cleanup_errors)); primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command); primary_tb = primary.__traceback__
    if primary is not None:
        record_stage(stage, "FAILED", returncode=returncode, seconds=time.monotonic()-started, command=command, reason=f"{type(primary).__name__}: {primary}", cleanup_errors=cleanup_errors)
        print(f"[{stage}] failed; full output: {LOG_PATH}", flush=True)
        raise primary.with_traceback(primary_tb)
    record_stage(stage, "SUCCEEDED" if returncode == 0 else "FAILED", returncode=returncode, seconds=time.monotonic()-started, command=command, cleanup_errors=cleanup_errors)
    print(f"[{stage}] returncode={returncode}; full output: {LOG_PATH}", flush=True)
    return returncode

def load_companion_source():
    required = PORTABLE_ROOT / "experiments/paper_results_v1/real_cli.py"
    archive_path = OUTPUT_ROOT / "companion_source.zip"
    status = "REUSED_EDITABLE_WORKING_DIRECTORY"
    if not required.is_file():
        status = "DOWNLOADED_AND_EXTRACTED"
        urllib.request.urlretrieve(COMPANION_ZIP_URL, archive_path)
        PORTABLE_ROOT.mkdir(parents=True, exist_ok=True)
        root_resolved = PORTABLE_ROOT.resolve()
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (PORTABLE_ROOT / member.filename).resolve()
                if root_resolved not in target.parents and target != root_resolved:
                    raise RuntimeError("unsafe companion archive member")
            archive.extractall(PORTABLE_ROOT)
    if not required.is_file():
        raise FileNotFoundError("companion source is missing experiments/paper_results_v1/real_cli.py")
    receipt = {
        "status": status,
        "source_repository": SOURCE_REPOSITORY,
        "source_ref": SOURCE_REF,
        "companion_zip_url": COMPANION_ZIP_URL,
        "source_root": str(PORTABLE_ROOT),
        "manifest_required": False,
        "digest_required": False,
        "git_clean_required": False,
    }
    atomic_json(OUTPUT_ROOT / "companion_source_receipt.json", receipt)
    return receipt

try:
    SOURCE_RECEIPT = load_companion_source()
    portable_source_path = str(PORTABLE_ROOT)
    if portable_source_path in sys.path:
        sys.path.remove(portable_source_path)
    sys.path.insert(0, portable_source_path)
    importlib.invalidate_caches()
    BASE_CONFIG = PORTABLE_ROOT / "experiments/paper_results_v1/real_eval.adopted.json"
    BOOTSTRAP_PLAN = OUTPUT_ROOT / "bootstrap_plan"
    portable_env = {**os.environ, "PYTHONPATH": str(PORTABLE_ROOT)}
    logged([PYTHON, "-u", "-m", "experiments.paper_results_v1.real_cli", "--config", str(BASE_CONFIG), "--output", str(BOOTSTRAP_PLAN), "--phase", "plan"], "BOOTSTRAP_FIXED_PLAN", cwd=PORTABLE_ROOT, env=portable_env)
    atomic_json(OUTPUT_ROOT / "execution_scope.json", {
        "status": "FIXED_EIGHT_SOURCE_CONFIRMATION_SCOPE",
        "attempted_case_ids": list(CONFIRMATION_CASES),
        "pilot_case_ids": ["pilot_01", "pilot_02"],
        "pilot_status": "NOT_EXECUTED_BY_NOTEBOOK",
        "reason": "Run all is fixed to the eight adopted confirmation cases; the two development pilots are not attempted or pooled into confirmation.",
        "full_denominator": FIXED_FULL_DENOMINATOR,
        "attempt_scope": FIXED_ATTEMPT_SCOPE,
    })
    record_stage("PORTABLE_SOURCE_AND_BOOTSTRAP_PLAN", "SUCCEEDED")
except Exception as exc:
    atomic_json(OUTPUT_ROOT / "bootstrap_failure.json", {"status": "FAILED", "reason": f"{type(exc).__name__}: {exc}", "traceback": traceback.format_exc(), "full_denominator": FIXED_FULL_DENOMINATOR})
    raise
print("Run:", RUN_ID, "companion source:", SOURCE_RECEIPT["source_root"])
print("Fixed execution scope:", CONFIRMATION_CASES, "; pilots are not executed by this notebook.")


In [ ]:
## Prepare fixed source/checkpoint declarations using ordinary local paths
# Baseline preparation failures are retained and do not prevent the main-method attempt.
VIDEOSEAL_COMMIT = "870ca7fb33578b90f14c602016b6c2788096226e"
VIDEOSEAL_REPO = "https://github.com/facebookresearch/videoseal.git"
VIDEOSEAL_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/videoseal/y_256b_img.pth"
RIVAGAN_COMMIT = "efffa72a4ca46d4d5051f6970c96424c2cdab441"
RIVAGAN_REPO = "https://github.com/DAI-Lab/RivaGAN.git"
RIVAGAN_WEIGHT_COMMIT = "4d9928350509f808b6b57d48d2f958aef811d332"
RIVAGAN_CHECKPOINT_URL = "https://raw.githubusercontent.com/Peachypie98/RivaGAN/4d9928350509f808b6b57d48d2f958aef811d332/model_weight/rivagan_32bit_model.pt"

source_checkout_receipts = {}
def prepared_checkout(label, url, commit, destination):
    destination = Path(destination)
    created_now = not destination.exists()
    if created_now:
        logged(["git", "clone", "--filter=blob:none", url, str(destination)], label + "_CLONE")
        has_commit = subprocess.run(
            ["git", "-C", str(destination), "cat-file", "-e", commit + "^{commit}"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False,
        ).returncode == 0
        if not has_commit:
            logged(["git", "-C", str(destination), "fetch", "origin", commit], label + "_FETCH", check=False)
        if logged(["git", "-C", str(destination), "checkout", "--detach", commit], label + "_CHECKOUT", check=False):
            raise RuntimeError(label + " checkout failed")
    if not destination.is_dir():
        raise FileNotFoundError(label + " source directory is unavailable: " + str(destination))
    source_checkout_receipts[label] = {
        "status": "CREATED_FROM_REQUESTED_REF" if created_now else "REUSED_EXISTING_WORKING_DIRECTORY",
        "source_url": url,
        "requested_ref": commit,
        "path": str(destination),
        "created_now": created_now,
    }
    return destination

def cached_download(label, url, destination):
    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        temporary = destination.with_suffix(destination.suffix + ".download")
        if temporary.exists(): temporary.unlink()
        urllib.request.urlretrieve(url, temporary)
        os.replace(temporary, destination)
    source_checkout_receipts[label + "_FILE"] = {
        "status": "AVAILABLE",
        "source_url": url,
        "path": str(destination),
    }
    return destination

baseline_setup = {}
VS_ROOT = CACHE_ROOT / "sources" / ("videoseal-" + VIDEOSEAL_COMMIT)
VS_CARD = VS_ROOT / "videoseal/cards/videoseal_1.0.yaml"
VS_WEIGHT = CACHE_ROOT / "checkpoints/videoseal/y_256b_img.pth"
try:
    prepared_checkout("VIDEOSEAL_SOURCE", VIDEOSEAL_REPO, VIDEOSEAL_COMMIT, VS_ROOT)
    VS_WEIGHT = cached_download("VIDEOSEAL_CHECKPOINT", VIDEOSEAL_CHECKPOINT_URL, VS_WEIGHT)
    if not VS_CARD.is_file():
        raise FileNotFoundError("VideoSeal card is unavailable: " + str(VS_CARD))
    baseline_setup["videoseal"] = {"status": "READY", "source_url": VIDEOSEAL_REPO, "requested_ref": VIDEOSEAL_COMMIT, "source_path": str(VS_ROOT), "card_path": str(VS_CARD), "checkpoint_path": str(VS_WEIGHT)}
except Exception as exc:
    baseline_setup["videoseal"] = {"status": "FAILED", "reason": f"{type(exc).__name__}: {exc}", "source_commit": VIDEOSEAL_COMMIT}

RIVA_ROOT = CACHE_ROOT / "sources" / ("rivagan-" + RIVAGAN_COMMIT)
RIVA_WEIGHT = CACHE_ROOT / "checkpoints/rivagan/rivagan_32bit_model.pt"
try:
    prepared_checkout("RIVAGAN_SOURCE", RIVAGAN_REPO, RIVAGAN_COMMIT, RIVA_ROOT)
    RIVA_WEIGHT = cached_download("RIVAGAN_CHECKPOINT", RIVAGAN_CHECKPOINT_URL, RIVA_WEIGHT)
    baseline_setup["rivagan"] = {
        "status": "READY", "source_url": RIVAGAN_REPO,
        "requested_ref": RIVAGAN_COMMIT, "source_path": str(RIVA_ROOT),
        "checkpoint_path": str(RIVA_WEIGHT), "checkpoint_source_ref": RIVAGAN_WEIGHT_COMMIT,
        "checkpoint_provenance": "Peachypie98 community 32-bit checkpoint; not DAI-Lab official weights",
    }
except Exception as exc:
    baseline_setup["rivagan"] = {"status": "FAILED", "reason": f"{type(exc).__name__}: {exc}", "source_commit": RIVAGAN_COMMIT, "checkpoint_source_commit": RIVAGAN_WEIGHT_COMMIT}
atomic_json(OUTPUT_ROOT / "baseline_setup_receipts.json", baseline_setup)

WAN_REVISION = "0fad780a534b6463e45facd96134c9f345acfa5b"
FRAMEWISE_REVISION = "31f26fdeee1355a5c34592e401dd41e45d25a493"
WAN_SNAPSHOT = CACHE_ROOT / "models/Wan2.1-T2V-1.3B-Diffusers" / WAN_REVISION
FRAMEWISE_SNAPSHOT = CACHE_ROOT / "models/sd-vae-ft-mse" / FRAMEWISE_REVISION
effective = json.loads(BASE_CONFIG.read_text(encoding="utf-8"))
effective["models"]["wan"]["local_snapshot_path"] = str(WAN_SNAPSHOT)
effective["models"]["framewise"]["local_snapshot_path"] = str(FRAMEWISE_SNAPSHOT)
effective["models"]["videoseal"].update({
    "source_root": str(VS_ROOT), "source_commit": VIDEOSEAL_COMMIT,
    "card_path": str(VS_CARD), "checkpoint_path": str(VS_WEIGHT),
    "checkpoint_download_status": "DOWNLOADED_OR_REUSED" if baseline_setup["videoseal"]["status"] == "READY" else "FAILED_NOT_AVAILABLE",
})
effective["models"]["rivagan"].update({
    "source_root": str(RIVA_ROOT), "source_commit": RIVAGAN_COMMIT,
    "checkpoint_path": str(RIVA_WEIGHT),
    "source_url": RIVAGAN_REPO,
    "checkpoint_source_url": RIVAGAN_CHECKPOINT_URL,
    "checkpoint_source_commit": RIVAGAN_WEIGHT_COMMIT,
    "checkpoint_download_status": "DOWNLOADED_OR_REUSED" if baseline_setup["rivagan"]["status"] == "READY" else "FAILED_NOT_AVAILABLE",
})
for model_name, optional_fields in {
    "videoseal": ("card_sha256", "checkpoint_sha256"),
    "rivagan": ("checkpoint_sha256",),
}.items():
    for field in optional_fields:
        effective["models"][model_name].pop(field, None)
EFFECTIVE_CONFIG = OUTPUT_ROOT / "effective_config.json"
atomic_json(EFFECTIVE_CONFIG, effective)
portable_env = {**os.environ, "PYTHONPATH": str(PORTABLE_ROOT)}
logged([PYTHON, "-u", "-m", "experiments.paper_results_v1.real_cli", "--config", str(EFFECTIVE_CONFIG), "--output", str(OUTPUT_ROOT / "fixed_plan"), "--phase", "plan"], "EFFECTIVE_FIXED_PLAN", cwd=PORTABLE_ROOT, env=portable_env)
if not (RUN_OUTPUT / "run_state.json").is_file():
    logged([PYTHON, "-u", "-m", "experiments.paper_results_v1.real_cli", "--config", str(EFFECTIVE_CONFIG), "--output", str(RUN_OUTPUT), "--phase", "init"], "RUN_STATE_INIT", cwd=PORTABLE_ROOT, env=portable_env)
else:
    record_stage("RUN_STATE_INIT", "REUSED_DURABLE_RUN_STATE", path=str(RUN_OUTPUT / "run_state.json"))
record_stage(
    "FIXED_IDENTITIES_AND_RUN_INIT", "SUCCEEDED",
    effective_config_path=str(EFFECTIVE_CONFIG),
)
print("Initialized fixed ten-case source plan. Run all will attempt confirmation only:", CONFIRMATION_CASES)


In [ ]:
## Prepare the recorded main environment and isolated baseline dependencies
from experiments.paper_results_v1.colab_orchestration import (
    ensure_baseline_interpreter,
    prepare_baseline_environment,
)

# The historical non-zero pip check is retained as a diagnostic and is not a blanket hard failure.
MAIN_RECOMMENDED_VERSIONS = {
    "torch": "2.11.0", "torchvision": "0.26.0", "diffusers": "0.39.0", "transformers": "4.57.6",
    "numpy": "2.1.3", "accelerate": "1.15.0", "safetensors": "0.8.0",
    "huggingface-hub": "0.36.2", "tokenizers": "0.22.2",
    "sentencepiece": "0.2.2", "ftfy": "6.3.1",
}
distribution_modules = {
    "torch": "torch", "torchvision": "torchvision", "diffusers": "diffusers",
    "transformers": "transformers", "numpy": "numpy", "accelerate": "accelerate",
    "safetensors": "safetensors", "huggingface-hub": "huggingface_hub",
    "tokenizers": "tokenizers", "sentencepiece": "sentencepiece", "ftfy": "ftfy",
}
import importlib.metadata as package_metadata
version_observation = {}
for distribution, recommended in MAIN_RECOMMENDED_VERSIONS.items():
    try:
        actual = package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        actual = None
    version_observation[distribution] = {
        "recommended_historical_version": recommended,
        "actual_version": actual,
        "status": "INSTALLED" if actual else "NOT_INSTALLED",
    }
atomic_json(OUTPUT_ROOT / "main_environment_version_observation.json", version_observation)
probe = "from diffusers import AutoencoderKL,AutoencoderKLWan,WanPipeline; import torch,sentencepiece,ftfy; print('MAIN_IMPORT_READY')"
main_environment_status = "READY"
try:
    if logged([PYTHON, "-c", probe], "MAIN_DEPENDENCY_PROBE", check=False):
        missing = [
            distribution for distribution, module in distribution_modules.items()
            if importlib.util.find_spec(module) is None
        ]
        logged(
            [
                PYTHON, "-m", "pip", "install",
                *[
                    f"{name}=={MAIN_RECOMMENDED_VERSIONS[name]}"
                    for name in (missing or list(distribution_modules))
                ],
            ],
            "MAIN_IMPORT_FAILURE_REPAIR",
        )
        logged([PYTHON, "-c", probe], "MAIN_DEPENDENCY_REPROBE")
    dependency_check_returncode = logged([PYTHON, "-m", "pip", "check"], "MAIN_PIP_CHECK_DIAGNOSTIC", check=False)
    (OUTPUT_ROOT / "environment_freeze.txt").write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True), encoding="utf-8")
except Exception as exc:
    main_environment_status = "FAILED"
    dependency_check_returncode = None
    record_stage("MAIN_ENVIRONMENT_PREPARATION", "FAILED", reason=f"{type(exc).__name__}: {exc}")

BASELINE_VERSION_RECOMMENDATIONS = OUTPUT_ROOT / "baseline_version_recommendations.json"
atomic_json(BASELINE_VERSION_RECOMMENDATIONS, {
    "status": "HISTORICAL_RECOMMENDATIONS_NOT_CONSTRAINTS",
    "main_versions": MAIN_RECOMMENDED_VERSIONS,
    "version_differences_are_blocking": False,
})
BASELINE_REPAIR_CONSTRAINTS = OUTPUT_ROOT / "baseline_repair_constraints.txt"
BASELINE_REPAIR_CONSTRAINTS.write_text(
    "".join(f"{name}=={version}\n" for name, version in MAIN_RECOMMENDED_VERSIONS.items()),
    encoding="utf-8",
)
baseline_pythons = {}
baseline_dependencies = {
    "videoseal": [
        "PyWavelets", "av", "calflops", "decord", "einops==0.8.2", "lpips",
        "omegaconf==2.3.0", "antlr4-python3-runtime==4.9.*", "PyYAML>=5.1.0",
        "opencv-python", "pandas==2.2.3", "pycocotools", "pytorch-msssim",
        "scikit-image", "scipy==1.16.3", "setuptools", "tensorboard",
        "timm==0.9.16", "tqdm", "transformers", "safetensors",
    ],
    "rivagan": ["torch-dct==0.1.5", "opencv-python", "pandas==2.2.3"],
}
for method in ("videoseal", "rivagan"):
    venv = Path("/content") / ("paper-results-v1-" + method + "-venv")
    vpython = venv / "bin/python"
    baseline_pythons[method] = str(vpython)
    baseline_setup[method]["environment_python"] = str(vpython)

    def ensure_method_interpreter():
        if baseline_setup[method]["status"] != "READY":
            raise RuntimeError(method + " source/checkpoint preparation failed")
        prepared_python, receipt = ensure_baseline_interpreter(
            method=method,
            venv=venv,
            main_python=PYTHON,
            run=logged,
        )
        if prepared_python != str(vpython):
            raise RuntimeError(method + " dedicated interpreter path changed")
        baseline_setup[method]["interpreter_preparation"] = receipt
        baseline_setup[method].setdefault("interpreter_preparation_history", []).append(receipt)
        return prepared_python

    def install_baseline_dependencies():
        # Resolve only after the actual entry import fails. The repair then
        # constrains the known historical main stack; those versions are
        # never a prerequisite when the existing imports already work.
        # The obsolete official RivaGAN requirements file is never installed.
        logged([
            PYTHON, "-m", "pip", "--python", str(vpython), "install",
            "--constraint", str(BASELINE_REPAIR_CONSTRAINTS),
            *baseline_dependencies[method],
        ], method.upper() + "_ISOLATED_DEPENDENCIES")
        baseline_setup[method]["environment_freeze_path"] = str(OUTPUT_ROOT / (method + "_environment_freeze.txt"))
        try:
            freeze = subprocess.check_output(
                [PYTHON, "-m", "pip", "--python", str(vpython), "freeze"],
                text=True,
            )
            (OUTPUT_ROOT / (method + "_environment_freeze.txt")).write_text(
                freeze, encoding="utf-8",
            )
            baseline_setup[method]["environment_freeze_status"] = "RECORDED"
        except Exception as exc:
            baseline_setup[method]["environment_freeze_status"] = "OBSERVATION_UNAVAILABLE"
            baseline_setup[method]["environment_freeze_reason"] = f"{type(exc).__name__}: {exc}"

    def probe_baseline_entry():
        if method == "videoseal":
            probe_code = (
                "import sys; sys.path.insert(0," + repr(str(VS_ROOT)) + "); "
                "from omegaconf import OmegaConf; "
                "from videoseal.utils.cfg import setup_model; "
                "print('VIDEOSEAL_IMPORT_READY_NO_MODEL_LOADED')"
            )
        else:
            probe_code = (
                "import sys; sys.path[:0]=" + repr([str(PORTABLE_ROOT), str(RIVA_ROOT)]) + "; "
                "from experiments.paper_results_v1.real_backends import _install_rivagan_pickle_classes; "
                "_install_rivagan_pickle_classes(); "
                "print('RIVAGAN_IMPORT_READY_NO_CHECKPOINT_LOADED')"
            )
        logged(
            [str(vpython), "-c", probe_code],
            method.upper() + "_ENTRY_IMPORT_PROBE", env=portable_env,
        )

    try:
        ensure_method_interpreter()
    except Exception as exc:
        baseline_setup[method].update(
            interpreter_preparation={
                "status": "FAILED",
                "reason": f"{type(exc).__name__}: {exc}",
                "ensurepip_used": False,
            },
            dependency_install_status="BLOCKED_INTERPRETER_PREPARATION_FAILED",
            entry_import_probe_status="BLOCKED_INTERPRETER_PREPARATION_FAILED",
            model_compatibility_status="NOT_VALIDATED_REQUIRES_REAL_EMBED_EXTRACT",
        )
        continue
    baseline_setup[method].update(prepare_baseline_environment(
        install=install_baseline_dependencies,
        probe=probe_baseline_entry,
    ))
atomic_json(OUTPUT_ROOT / "baseline_setup_receipts.json", baseline_setup)

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token: os.environ["HF_TOKEN"] = hf_token
except Exception:
    hf_token = None
try:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Wan-AI/Wan2.1-T2V-1.3B-Diffusers", revision=WAN_REVISION,
        local_dir=str(WAN_SNAPSHOT),
        allow_patterns=["model_index.json", "scheduler/*", "tokenizer/*", "text_encoder/*", "transformer/*", "vae/*"],
    )
    snapshot_download(
        repo_id="stabilityai/sd-vae-ft-mse", revision=FRAMEWISE_REVISION,
        local_dir=str(FRAMEWISE_SNAPSHOT),
        allow_patterns=["config.json", "diffusion_pytorch_model.safetensors"],
    )
    model_snapshot_status = "READY"
except Exception as exc:
    model_snapshot_status = "FAILED"
    record_stage("MODEL_SNAPSHOT_PREPARATION", "FAILED", reason=f"{type(exc).__name__}: {exc}")
finally:
    hf_token = None
atomic_json(OUTPUT_ROOT / "environment_setup_receipt.json", {
    "main_environment_status": main_environment_status,
    "main_pip_check_returncode_nonfatal_diagnostic": dependency_check_returncode,
    "model_snapshot_status": model_snapshot_status,
    "wan_revision": WAN_REVISION, "framewise_revision": FRAMEWISE_REVISION,
    "baseline_version_recommendations_path": str(BASELINE_VERSION_RECOMMENDATIONS),
    "baseline_repair_constraints_path": str(BASELINE_REPAIR_CONSTRAINTS),
    "baseline_environments": baseline_setup,
    "note": "Model/backend compatibility is established only by later phase receipts, not by preparation status.",
})
print("Environment:", main_environment_status, "models:", model_snapshot_status, "pip check diagnostic:", dependency_check_returncode)


In [ ]:
# Initialize the adopted confirmation attack and receiver plans.
ATTACK_CONFIG = OUTPUT_ROOT / "effective_confirmation_config.json"
attack_cfg = json.loads((PORTABLE_ROOT / "experiments/paper_results_v1/confirmation_v1.adopted.json").read_text())
attack_cfg["source_run"]["run_dir"] = str(RUN_OUTPUT)
attack_cfg["models"]["wan"]["local_snapshot_path"] = str(WAN_SNAPSHOT)
attack_cfg["models"]["framewise"]["local_snapshot_path"] = str(FRAMEWISE_SNAPSHOT)
attack_cfg["models"]["videoseal"].update(source_root=str(VS_ROOT), card_path=str(VS_CARD), checkpoint_path=str(VS_WEIGHT))
attack_cfg["models"]["rivagan"].update(source_root=str(RIVA_ROOT), checkpoint_path=str(RIVA_WEIGHT))
for model_name, optional_fields in {"videoseal": ("card_sha256", "checkpoint_sha256"), "rivagan": ("checkpoint_sha256",)}.items():
    for field in optional_fields: attack_cfg["models"][model_name].pop(field, None)
atomic_json(ATTACK_CONFIG, attack_cfg)
ATTACK_OUTPUT = OUTPUT_ROOT / "attack_run"
ATTACK_TEMP = Path("/content") / ("paper-results-v1-confirmation-attack-" + RUN_ID)
CONFIRM_OUTPUT = OUTPUT_ROOT / "confirmation_receiver"
ATTACK_CLI = [PYTHON, "-u", "-m", "experiments.paper_results_v1.attack_cli", "--config", str(ATTACK_CONFIG)]
CONFIRM_CLI = [PYTHON, "-u", "-m", "experiments.paper_results_v1.confirmation_v1_cli", "--config", str(ATTACK_CONFIG)]
logged([*ATTACK_CLI, "--output", str(OUTPUT_ROOT / "attack_plan.json"), "--phase", "plan"], "ATTACK_FIXED_PLAN", cwd=PORTABLE_ROOT, env=portable_env)
if not (ATTACK_OUTPUT / "attack_run_state.json").is_file():
    logged([*ATTACK_CLI, "--output", str(ATTACK_OUTPUT), "--temp-root", str(ATTACK_TEMP), "--phase", "init"], "ATTACK_RUN_INIT", cwd=PORTABLE_ROOT, env=portable_env)
else:
    record_stage("ATTACK_RUN_INIT", "REUSED_DURABLE_RUN_STATE", path=str(ATTACK_OUTPUT / "attack_run_state.json"))
logged([*CONFIRM_CLI, "--output", str(OUTPUT_ROOT / "confirmation_plan.json"), "--phase", "plan"], "CONFIRMATION_FIXED_PLAN", cwd=PORTABLE_ROOT, env=portable_env)
if not (CONFIRM_OUTPUT / "confirmation_state.json").is_file():
    logged([*CONFIRM_CLI, "--output", str(CONFIRM_OUTPUT), "--attack-output", str(ATTACK_OUTPUT), "--temp-root", str(ATTACK_TEMP), "--phase", "init"], "CONFIRMATION_RUN_INIT", cwd=PORTABLE_ROOT, env=portable_env)
else:
    record_stage("CONFIRMATION_RUN_INIT", "REUSED_DURABLE_RUN_STATE", path=str(CONFIRM_OUTPUT / "confirmation_state.json"))


In [ ]:
# Execute each model family in a separate process. Failures are retained and later cases continue.
ATTEMPTS_PATH = OUTPUT_ROOT / "phase_attempts.json"
attempts = json.loads(ATTEMPTS_PATH.read_text()) if ATTEMPTS_PATH.is_file() else []
GENERATION_PHASES = ("generate", "decode", "framewise", "codec")
ATTACK_BEFORE_READ = (
    "import-main", "baseline-embed-videoseal", "baseline-embed-rivagan",
    "baseline-codec", "attack-media", "receiver-clock",
)
ATTACK_AFTER_READ = ("baseline-extract-videoseal", "baseline-extract-rivagan", "quality")
ATTACK_PHASES = (*ATTACK_BEFORE_READ, *ATTACK_AFTER_READ)
interpreters = {
    "baseline-embed-videoseal": baseline_pythons["videoseal"],
    "baseline-extract-videoseal": baseline_pythons["videoseal"],
    "baseline-embed-rivagan": baseline_pythons["rivagan"],
    "baseline-extract-rivagan": baseline_pythons["rivagan"],
}
def retain_attempt(case_id, family, phase, status, **fields):
    attempts.append({"case_id": case_id, "family": family, "phase": phase, "status": status, **fields})
    atomic_json(ATTEMPTS_PATH, attempts)
def phase_status(family, case_id, phase):
    try:
        if family == "generation":
            state = json.loads((RUN_OUTPUT / "run_state.json").read_text())
            return state["phases"][phase]["cases"][case_id]["status"]
        if family == "attack":
            state = json.loads((ATTACK_OUTPUT / "attack_run_state.json").read_text())
            return state["phases"][phase]["cases"][case_id]["status"]
        state = json.loads((CONFIRM_OUTPUT / "confirmation_state.json").read_text())
        return state["phases"][phase][case_id]["status"]
    except Exception:
        return None
def invoke_owned(command, case_id, family, phase, *, interpreter=PYTHON):
    prior = phase_status(family, case_id, phase)
    if prior == "COMPLETE":
        retain_attempt(case_id, family, phase, "SKIPPED_DURABLE_COMPLETE")
        return 0
    if family in ("generation", "attack") and prior not in (None, "PLANNED"):
        retain_attempt(case_id, family, phase, "SKIPPED_RETAINED_" + str(prior))
        return None
    return invoke(command, case_id, family, phase, interpreter=interpreter)
def invoke(command, case_id, family, phase, *, interpreter=PYTHON):
    command = [interpreter, *command]
    try:
        rc = logged(command, f"{case_id}:{family}:{phase}", cwd=PORTABLE_ROOT, env=portable_env, check=False)
        retain_attempt(case_id, family, phase, "COMPLETE" if rc == 0 else "FAILED_RETAINED", returncode=rc)
        return rc
    except KeyboardInterrupt:
        retain_attempt(case_id, family, phase, "INTERRUPTED", reason="KeyboardInterrupt")
        raise
    except BaseException as exc:
        retain_attempt(case_id, family, phase, "FAILED_RETAINED", reason=f"{type(exc).__name__}: {exc}")
        return None

interrupted = None
real_prefix = ["-u", "-m", "experiments.paper_results_v1.real_cli", "--config", str(EFFECTIVE_CONFIG), "--output", str(RUN_OUTPUT)]
attack_prefix = ["-u", "-m", "experiments.paper_results_v1.attack_cli", "--config", str(ATTACK_CONFIG), "--output", str(ATTACK_OUTPUT), "--temp-root", str(ATTACK_TEMP)]
confirm_prefix = ["-u", "-m", "experiments.paper_results_v1.confirmation_v1_cli", "--config", str(ATTACK_CONFIG), "--output", str(CONFIRM_OUTPUT), "--temp-root", str(ATTACK_TEMP)]
try:
    invoke([*real_prefix, "--phase", "preflight"], None, "generation", "preflight")
    for case_id in CONFIRMATION_CASES:
        for phase in GENERATION_PHASES:
            invoke_owned([*real_prefix, "--phase", phase, "--case-id", case_id], case_id, "generation", phase)
    for case_id in CONFIRMATION_CASES:
        for phase in ATTACK_BEFORE_READ:
            interpreter = interpreters.get(phase, PYTHON)
            invoke_owned([*attack_prefix, "--phase", phase, "--case-id", case_id], case_id, "attack", phase, interpreter=interpreter)
    for case_id in CONFIRMATION_CASES:
        for phase in ("prepare", "read"):
            invoke_owned([*confirm_prefix, "--phase", phase, "--case-id", case_id], case_id, "confirmation", phase)
    for case_id in CONFIRMATION_CASES:
        for phase in ATTACK_AFTER_READ:
            interpreter = interpreters.get(phase, PYTHON)
            invoke_owned([*attack_prefix, "--phase", phase, "--case-id", case_id], case_id, "attack", phase, interpreter=interpreter)
    invoke([*attack_prefix, "--phase", "evaluate"], None, "attack", "evaluate")
    invoke([*confirm_prefix, "--phase", "report"], None, "confirmation", "report")
except KeyboardInterrupt as exc:
    interrupted = f"{type(exc).__name__}: {exc}"
    record_stage("CONFIRMATION_SEQUENCE", "INTERRUPTED", reason=interrupted)
    try:
        invoke([*attack_prefix, "--phase", "evaluate"], None, "attack", "evaluate-after-interrupt")
        invoke([*confirm_prefix, "--phase", "report"], None, "confirmation", "report-after-interrupt")
    finally:
        atomic_json(OUTPUT_ROOT / "interruption_handoff.json", {"status": "INTERRUPTED", "reason": interrupted, "attempts": attempts})
    raise
record_stage("CONFIRMATION_SEQUENCE", "COMPLETE_WITH_RETAINED_ITEM_STATUSES", attempts=len(attempts))


In [ ]:
# Compact handoff; status counts come from final retained rows, not process return codes.
def optional_json(path):
    try: return json.loads(Path(path).read_text())
    except Exception as exc: return {"status": "UNAVAILABLE", "reason": f"{type(exc).__name__}: {exc}"}
confirmation_report = optional_json(CONFIRM_OUTPUT / "evaluation_report.json")
attack_report = optional_json(ATTACK_OUTPUT / "evaluation_report.json")
confirmation_baselines = [row for row in attack_report.get("baseline_rows", []) if row.get("case_id") in CONFIRMATION_CASES]
confirmation_quality = [row for row in attack_report.get("quality_rows", []) if row.get("case_id") in CONFIRMATION_CASES]
handoff = {
    "schema": "paper-results-v1-confirmation-v1-handoff",
    "run_id": RUN_ID, "execution_scope": list(CONFIRMATION_CASES),
    "pilots": {"status": "NOT_EXECUTED_BY_NOTEBOOK", "case_ids": ["pilot_01", "pilot_02"]},
    "method": {"preferred_receiver": "CENTERED_U", "controls": ["RAW_U", "CENTERED_D4"]},
    "fixed_denominator": FIXED_ATTEMPT_SCOPE,
    "confirmation_report_status": confirmation_report.get("status"),
    "receiver_status_counts": confirmation_report.get("receiver_status_counts"),
    "baseline_status_counts": ({row.get("status"): sum(1 for x in confirmation_baselines if x.get("status") == row.get("status")) for row in confirmation_baselines}),
    "quality_status_counts": ({row.get("status"): sum(1 for x in confirmation_quality if x.get("status") == row.get("status")) for row in confirmation_quality}),
    "attempts": attempts,
    "files": {
        "generation_state": str(RUN_OUTPUT / "run_state.json"),
        "attack_state": str(ATTACK_OUTPUT / "attack_run_state.json"),
        "attack_report": str(ATTACK_OUTPUT / "evaluation_report.json"),
        "confirmation_state": str(CONFIRM_OUTPUT / "confirmation_state.json"),
        "confirmation_report": str(CONFIRM_OUTPUT / "evaluation_report.json"),
        "receiver_csv": str(CONFIRM_OUTPUT / "receiver_rows.csv"),
        "execution_log": str(LOG_PATH), "stage_receipts": str(STAGE_RECEIPTS),
    },
    "evidence_ceiling": "Eight predeclared source-level clusters under the fixed method; no independence, unseen, FPR, or threshold claim.",
}
atomic_json(OUTPUT_ROOT / "handoff_summary.json", handoff)
print("Handoff:", OUTPUT_ROOT / "handoff_summary.json")
print("Receiver status counts:", handoff["receiver_status_counts"])


## Return the complete unique Drive directory

Return `handoff_summary.json`, all three run-state/report trees, CSV files, NPZ sidecars, media receipts, `execution.log`, and `stage_receipts.json`. The deterministic plan comprises 16 Wan trajectories, 8 shared writer encodes, 16 writer decodes, 48 initial and 672 temporal-edit codec roundtrips, 480 clock encodes, 2,880 logical receiver rows, 240 whole-video baseline extract entries, 56 quality rows, and 16 display codecs. RivaGAN's backend expands its eight embeds into 1,448 frame encoder calls and its 120 extracts into 21,608 frame decoder calls; VideoSeal uses eight video-level embeds and 120 video-level detects. These are planned calls, not measured time or GPU memory.

Quality covers 1,288 LPIPS batch forwards and 10,080 Farneback transitions. Wan physical receiver encodes are recorded from actual calls and can range from zero to 2,880; 480 is only a conditional shared-map lower bound. K0/K1 may share a VAE latent for the same artifact/map, but their FFT coordinates, votes, decoded bits, and durable records remain separate.

This notebook and companion were validated locally with CPU/static/stub checks only. No real confirmation model, codec, GPU, Colab, or Drive execution occurred during delivery. Peak GPU memory and wall time remain unmeasured.
